# Plan 1: Supervoxel GNN — Binary Tumor Refinement + Edge Prediction + Explainability**Research Question:** Can a GNN, given the segmentation model's output as a prior, correct segmentation errors by reasoning over the graph neighborhood?## Architecture (~885K parameters, ~3.5 MB)```Patch Tensor (16×19) → PatchEmbedder (Transformer 3L×4H) → 128-dim    → + Laplacian PE (8-dim) → GraphEncoder (GATv2 3L×4H) → 128-dim    ├→ NodeHead → binary tumor logit (Task 1)    └→ EdgeHead → 10-class boundary type (Task 2)```**Explainability:** 3-level attention traces (graph, patch, refinement)

In [ ]:
!pip install -q torch==2.1.2 torchvision==0.16.2 --index-url https://download.pytorch.org/whl/cu118
!pip install -q torch-geometric scikit-image nibabel

## Configuration

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import random
import time
from pathlib import Path

SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

SHARED = {
    "seed": SEED,
    "device": DEVICE,
    "img_size": 224,
    "checkpoint_dir": Path("checkpoints"),
}

GRAPH = {
    "data_root": Path("/kaggle/input/brats2023-gli/BraTS-GLI"),
    "modalities": ["t1n", "t1c", "t2w", "t2f"],
    "slic_modality": "t1n",
    "num_classes": 4,
    "class_names": {0: "BG", 1: "NCR", 2: "ED", 3: "ET"},
    "n_segments": 1000,
    "compactness": 0.1,
    "min_sv_volume": 20,
    "tau": 0.15,
    "n_patch": 4,
    "patch_neighbors": 16,
    "knn_k": 8,
    "embed_dim": 128,
    "transformer_layers": 3,
    "transformer_heads": 4,
    "gat_layers": 3,
    "gat_heads": 4,
    "laplacian_pe_dim": 8,
    "epochs": 100,
    "lr": 5e-4,
    "weight_decay": 0.01,
    "batch_size": 2,
    "accum_steps": 4,
    "eval_every": 5,
    "num_folds": 5,
    "lambda_node": 1.0,
    "lambda_edge": 0.5,
    "checkpoint": Path("checkpoints/graph_plan1.pth"),
    "cache_dir": Path("graph_cache"),
}

print(f"Device: {DEVICE}")
print(f"Config loaded: embed_dim={GRAPH['embed_dim']}, n_segments={GRAPH['n_segments']}")

## Supervoxel Preprocessing (Steps 1–7)Converts native-resolution BraTS 3D MRI volumes into supervoxel graph data:1. 3D SLIC clustering on T1n2. Background pruning (largest-gap heuristic)3. Per-SV ground truth (tumor proportion, binary label, dominant class)4. Seg prior features (optional, from frozen DeepLabV3+)5. Patch extraction (k-means++ centroids, multi-modal)6. kNN graph construction7. Edge-level ground truth (10-class boundary types)

In [ ]:
import nibabel as nib
from skimage.segmentation import slic
from sklearn.cluster import KMeans
from scipy.spatial import cKDTree



# ── Data Loading (native resolution) ─────────────────────────────────


def discover_cases(data_dir):
    data_dir = Path(data_dir)
    cases = sorted([d for d in data_dir.iterdir() if d.is_dir()])
    valid = []

    for case_dir in cases:
        case_id = case_dir.name
        files = {}
        all_found = True

        for key in [*GRAPH["modalities"], "seg"]:
            # Try both .nii and .nii.gz
            nii = case_dir / f"{case_id}-{key}.nii"
            nii_gz = case_dir / f"{case_id}-{key}.nii.gz"
            if nii.exists():
                files[key] = nii
            elif nii_gz.exists():
                files[key] = nii_gz
            else:
                all_found = False
                break

        if all_found:
            valid.append({"case_id": case_id, "files": files})

    return valid


def load_volume(nii_path):
    """Load a NIfTI volume at native resolution as float32."""
    return nib.load(str(nii_path)).get_fdata().astype(np.float32)


def zscore_normalize(volume):
    """Z-score normalize non-zero voxels (standard BraTS preprocessing)."""
    out = volume.copy()
    mask = out > 0
    if mask.sum() == 0:
        return out
    mean = out[mask].mean()
    std = out[mask].std()
    if std < 1e-8:
        return out
    out[mask] = (out[mask] - mean) / std
    return out


def load_case_volumes(case):
    modality_vols = {}
    for mod in GRAPH["modalities"]:
        vol = load_volume(case["files"][mod])
        modality_vols[mod] = zscore_normalize(vol)

    gt_seg = load_volume(case["files"]["seg"]).astype(np.int32)
    shape = modality_vols[GRAPH["slic_modality"]].shape

    return modality_vols, gt_seg, shape


# ── Step 1: 3D SLIC Supervoxel Generation ────────────────────────────


def generate_supervoxels(t1n_volume, n_segments=None, compactness=None):
    n_segments = n_segments or GRAPH["n_segments"]
    compactness = compactness or GRAPH["compactness"]

    sv_labels = slic(
        t1n_volume,
        n_segments=n_segments,
        compactness=compactness,
        start_label=0,
        enforce_connectivity=True,
        channel_axis=None,          # input is single-channel 3D
    )

    return sv_labels.astype(np.int32)


# ── Step 2: Dynamic Background Pruning ───────────────────────────────


def prune_background(sv_labels, t1n_volume, min_volume=None):
    min_volume = min_volume or GRAPH["min_sv_volume"]
    unique_labels = np.unique(sv_labels)

    # Compute per-SV statistics
    sv_stats = {}
    for label in unique_labels:
        mask = sv_labels == label
        voxels = t1n_volume[mask]
        sv_stats[label] = {
            "mean_intensity": float(voxels.mean()),
            "volume": int(mask.sum()),
        }

    # Filter by minimum volume first
    volume_ok = {
        l for l, s in sv_stats.items()
        if s["volume"] >= min_volume
    }

    # Sort remaining SVs by mean intensity (ascending)
    sorted_labels = sorted(volume_ok, key=lambda l: sv_stats[l]["mean_intensity"])
    sorted_means = [sv_stats[l]["mean_intensity"] for l in sorted_labels]

    if len(sorted_means) < 2:
        # Edge case: can't compute gaps with fewer than 2 SVs
        return list(volume_ok), {
            "theta": 0.0,
            "total_svs": len(unique_labels),
            "retained": len(volume_ok),
            "pruned_bg": 0,
            "pruned_small": len(unique_labels) - len(volume_ok),
        }

    # Find largest gap in sorted mean distribution
    gaps = np.diff(sorted_means)
    g = int(np.argmax(gaps))
    theta = 0.5 * (sorted_means[g] + sorted_means[g + 1])

    # Retain SVs with mean intensity above threshold
    retained = [l for l in sorted_labels if sv_stats[l]["mean_intensity"] > theta]
    pruned_bg = len(sorted_labels) - len(retained)
    pruned_small = len(unique_labels) - len(volume_ok)

    pruning_info = {
        "theta": float(theta),
        "total_svs": len(unique_labels),
        "after_volume_filter": len(volume_ok),
        "retained": len(retained),
        "pruned_bg": pruned_bg,
        "pruned_small": pruned_small,
        "sv_stats": sv_stats,
    }

    return retained, pruning_info


# ── Step 3: Supervoxel-Level Ground Truth ────────────────────────────


def compute_sv_targets(sv_labels, gt_seg, retained_labels, tau=None):
    tau = tau if tau is not None else GRAPH["tau"]
    num_classes = GRAPH["num_classes"]

    # Pre-compute voxel coordinate arrays for centroid calculation
    coords = np.array(np.where(sv_labels >= 0))  # (3, total_voxels)

    targets = {}
    tumor_count = 0
    healthy_count = 0

    for label in retained_labels:
        mask = sv_labels == label
        gt_voxels = gt_seg[mask].astype(np.int64)
        total = gt_voxels.size

        # ── Tumor proportion (any class > 0 is tumor) ──
        n_tumor = int((gt_voxels > 0).sum())
        y_reg = n_tumor / total if total > 0 else 0.0

        # ── Binary tumor label ──
        y_cls = 1 if y_reg > tau else 0

        # ── Dominant class (mode) ──
        counts = np.bincount(gt_voxels, minlength=num_classes)
        y_dominant = int(counts.argmax())

        # ── Class proportions vector ──
        y_props = (counts / total).astype(np.float32) if total > 0 else np.zeros(num_classes, dtype=np.float32)

        # ── Centroid (mean voxel coordinate) ──
        sv_coords = np.argwhere(mask)  # (N_voxels, 3)
        centroid = sv_coords.mean(axis=0).astype(np.float32)  # (3,)

        targets[label] = {
            "y_reg": float(y_reg),
            "y_cls": int(y_cls),
            "y_dominant": int(y_dominant),
            "y_props": y_props,
            "volume": int(total),
            "centroid": centroid,
            "class_counts": counts.astype(np.int64),
        }

        if y_cls == 1:
            tumor_count += 1
        else:
            healthy_count += 1

    summary = {
        "total_svs": len(retained_labels),
        "tumor_svs": tumor_count,
        "healthy_svs": healthy_count,
        "tumor_ratio": tumor_count / len(retained_labels) if retained_labels else 0.0,
        "mean_tumor_proportion": float(np.mean([t["y_reg"] for t in targets.values()])),
    }

    return targets, summary


# ── Step 4: Segmentation Prior Features ──────────────────────────────


def load_seg_probabilities(case_id, prob_dir=None):
    if prob_dir is None:
        prob_dir = Path("brats_outputs/seg_probs")
    else:
        prob_dir = Path(prob_dir)

    prob_path = prob_dir / f"{case_id}_seg_probs.npy"
    if not prob_path.exists():
        return None

    return np.load(str(prob_path))


def compute_seg_prior_features(sv_labels, retained_labels, seg_probs):
    num_classes = seg_probs.shape[0]
    seg_features = {}

    all_entropies = []

    for label in retained_labels:
        mask = sv_labels == label

        # Mean probability vector across all voxels in this SV
        # seg_probs[:, mask] has shape (num_classes, n_voxels)
        seg_feat = seg_probs[:, mask].mean(axis=1).astype(np.float32)  # (4,)

        # Prediction entropy: -Σ p_c log(p_c)  (clipped to avoid log(0))
        p_clipped = np.clip(seg_feat, 1e-8, 1.0)
        seg_entropy = float(-np.sum(p_clipped * np.log(p_clipped)))

        # Predicted class from mean probabilities
        seg_pred = int(seg_feat.argmax())

        seg_features[label] = {
            "seg_feat": seg_feat,
            "seg_entropy": seg_entropy,
            "seg_pred": seg_pred,
        }

        all_entropies.append(seg_entropy)

    # Summary statistics
    entropies = np.array(all_entropies)
    pred_counts = {}
    for sf in seg_features.values():
        c = sf["seg_pred"]
        pred_counts[c] = pred_counts.get(c, 0) + 1

    seg_summary = {
        "mean_entropy": float(entropies.mean()),
        "std_entropy": float(entropies.std()),
        "max_entropy": float(entropies.max()),
        "high_uncertainty_svs": int((entropies > 1.0).sum()),  # entropy > 1.0 nats
        "pred_class_distribution": pred_counts,
    }

    return seg_features, seg_summary

In [ ]:
# ── Step 5: Patch Extraction ─────────────────────────────────────────


def extract_patches(sv_labels, modality_vols, retained_labels, targets,
                    n_patch=None, patch_neighbors=None):
    n_patch = n_patch or GRAPH["n_patch"]
    s = patch_neighbors or GRAPH["patch_neighbors"]
    mod_names = list(modality_vols.keys())
    n_mods = len(mod_names)

    patches = {}
    padded_count = 0

    for label in retained_labels:
        voxel_coords = np.argwhere(sv_labels == label)  # (N_vox, 3)
        N = voxel_coords.shape[0]

        # k-means++ centroid selection
        actual_k = min(n_patch, N)
        if actual_k < 2:
            centroids = voxel_coords[:actual_k].astype(np.float64)
        else:
            km = KMeans(
                n_clusters=actual_k, init="k-means++",
                n_init=1, max_iter=20, random_state=42,
            )
            km.fit(voxel_coords)
            centroids = km.cluster_centers_  # (actual_k, 3)

        # Build KDTree for nearest-neighbor queries within this SV
        tree = cKDTree(voxel_coords)

        rows = []
        for ci in range(actual_k):
            centroid = centroids[ci]
            k_query = min(s, N)
            _, nn_idx = tree.query(centroid, k=k_query)
            nn_idx = np.atleast_1d(nn_idx)
            nn_coords = voxel_coords[nn_idx]  # (k_query, 3)

            for mod in mod_names:
                vol = modality_vols[mod]
                values = vol[nn_coords[:, 0], nn_coords[:, 1], nn_coords[:, 2]]

                # Pad if fewer neighbors than s
                if len(values) < s:
                    values = np.pad(values, (0, s - len(values)), mode="edge")
                    padded_count += 1

                # Augment with centroid XYZ
                row = np.concatenate([values, centroid])  # (s + 3,)
                rows.append(row)

        # Pad missing centroids if SV was very small
        while len(rows) < n_patch * n_mods:
            rows.append(rows[-1].copy())
            padded_count += 1

        patches[label] = np.stack(rows, axis=0).astype(np.float32)

    patch_info = {
        "n_patch": n_patch,
        "patch_neighbors": s,
        "n_modalities": n_mods,
        "tensor_shape": f"({n_patch * n_mods}, {s + 3})",
        "padded_patches": padded_count,
    }

    return patches, patch_info


# ── Step 6: kNN Graph Construction ───────────────────────────────────


def build_knn_graph(retained_labels, targets, k=None):
    k = k or GRAPH["knn_k"]
    N = len(retained_labels)

    # Map SV labels to consecutive node indices
    label_to_idx = {label: i for i, label in enumerate(retained_labels)}
    idx_to_label = {i: label for label, i in label_to_idx.items()}

    # Collect centroids in index order
    centroids = np.stack(
        [targets[retained_labels[i]]["centroid"] for i in range(N)],
        axis=0,
    )  # (N, 3)

    # kNN via KDTree
    actual_k = min(k + 1, N)  # +1 because query includes self
    tree = cKDTree(centroids)
    dists, indices = tree.query(centroids, k=actual_k)  # (N, actual_k)

    # Build edge list (skip self-loops at index 0)
    src_list = []
    dst_list = []
    attr_list = []

    for i in range(N):
        for j_pos in range(1, actual_k):  # skip self
            j = indices[i, j_pos]
            delta = centroids[j] - centroids[i]  # (3,)
            dist = dists[i, j_pos]

            src_list.append(i)
            dst_list.append(j)
            attr_list.append([delta[0], delta[1], delta[2], dist])

    # Symmetrize: add reverse edges (deduplicated)
    edge_set = set()
    sym_src, sym_dst, sym_attr = [], [], []
    for idx in range(len(src_list)):
        s, d = src_list[idx], dst_list[idx]
        if (s, d) not in edge_set:
            edge_set.add((s, d))
            sym_src.append(s)
            sym_dst.append(d)
            sym_attr.append(attr_list[idx])
        if (d, s) not in edge_set:
            edge_set.add((d, s))
            rev_delta = [-attr_list[idx][0], -attr_list[idx][1],
                         -attr_list[idx][2], attr_list[idx][3]]
            sym_src.append(d)
            sym_dst.append(s)
            sym_attr.append(rev_delta)

    if len(sym_src) > 0:
        edge_index = np.array([sym_src, sym_dst], dtype=np.int64)   # (2, E)
        edge_attr = np.array(sym_attr, dtype=np.float32).reshape(-1, 4)  # (E, 4)
    else:
        edge_index = np.zeros((2, 0), dtype=np.int64)
        edge_attr = np.zeros((0, 4), dtype=np.float32)

    # Degree statistics
    degrees = np.bincount(edge_index[0] if edge_index.shape[1] > 0 else [], minlength=N)

    graph_info = {
        "num_nodes": N,
        "num_edges": edge_index.shape[1],
        "k": k,
        "mean_degree": float(degrees.mean()),
        "min_degree": int(degrees.min()) if len(degrees) > 0 else 0,
        "max_degree": int(degrees.max()) if len(degrees) > 0 else 0,
        "avg_edge_dist": float(edge_attr[:, 3].mean()) if edge_attr.shape[0] > 0 else 0.0,
    }

    return edge_index, edge_attr, label_to_idx, idx_to_label, graph_info


# ── Step 7: Edge-Level Ground Truth ──────────────────────────────────

# Symmetric boundary type encoding: ordered pair (min, max) of class IDs
# 10 types for 4 classes: (0,0),(0,1),(0,2),(0,3),(1,1),(1,2),(1,3),(2,2),(2,3),(3,3)
BOUNDARY_TYPE_MAP = {}
_bt_idx = 0
for _a in range(4):
    for _b in range(_a, 4):
        BOUNDARY_TYPE_MAP[(_a, _b)] = _bt_idx
        _bt_idx += 1
BOUNDARY_TYPE_NAMES = {
    v: f"{GRAPH['class_names'][a]}↔{GRAPH['class_names'][b]}"
    for (a, b), v in BOUNDARY_TYPE_MAP.items()
}


def compute_edge_targets(edge_index, targets, idx_to_label):
    E = edge_index.shape[1]
    y_binary = np.zeros(E, dtype=np.int64)
    y_type = np.zeros(E, dtype=np.int64)
    y_grad = np.zeros(E, dtype=np.float32)

    for e in range(E):
        i_idx, j_idx = int(edge_index[0, e]), int(edge_index[1, e])
        label_i, label_j = idx_to_label[i_idx], idx_to_label[j_idx]

        dom_i = targets[label_i]["y_dominant"]
        dom_j = targets[label_j]["y_dominant"]
        reg_i = targets[label_i]["y_reg"]
        reg_j = targets[label_j]["y_reg"]

        # Binary: same class?
        y_binary[e] = 1 if dom_i == dom_j else 0

        # Boundary type: symmetric ordered pair
        pair = (min(dom_i, dom_j), max(dom_i, dom_j))
        y_type[e] = BOUNDARY_TYPE_MAP[pair]

        # Transition gradient
        y_grad[e] = abs(reg_i - reg_j)

    # Statistics
    type_counts = {}
    for t in y_type:
        name = BOUNDARY_TYPE_NAMES[int(t)]
        type_counts[name] = type_counts.get(name, 0) + 1

    edge_target_info = {
        "num_edges": E,
        "same_class_edges": int(y_binary.sum()),
        "diff_class_edges": int((1 - y_binary).sum()),
        "boundary_type_distribution": type_counts,
        "mean_gradient": float(y_grad.mean()),
        "max_gradient": float(y_grad.max()),
    }

    edge_targets = {
        "y_edge_binary": y_binary,
        "y_edge_type": y_type,
        "y_edge_grad": y_grad,
    }

    return edge_targets, edge_target_info


# ── Full Pipeline (Steps 1–7) ────────────────────────────────────────


def preprocess_case(case, config=None, seg_prob_dir=None):
    cfg = config or GRAPH
    case_id = case["case_id"]

    # Load native-resolution volumes
    modality_vols, gt_seg, shape = load_case_volumes(case)
    t1n = modality_vols[cfg["slic_modality"]]

    # Step 1: 3D SLIC
    sv_labels = generate_supervoxels(
        t1n, n_segments=cfg["n_segments"], compactness=cfg["compactness"]
    )

    # Step 2: Dynamic background pruning
    retained_labels, pruning_info = prune_background(
        sv_labels, t1n, min_volume=cfg["min_sv_volume"]
    )

    # Step 3: Ground truth
    targets, target_summary = compute_sv_targets(
        sv_labels, gt_seg, retained_labels, tau=cfg["tau"]
    )

    # Step 4: Segmentation prior features (optional)
    seg_features = None
    seg_summary = None
    if seg_prob_dir is not None:
        seg_probs = load_seg_probabilities(case_id, seg_prob_dir)
        if seg_probs is not None:
            seg_features, seg_summary = compute_seg_prior_features(
                sv_labels, retained_labels, seg_probs
            )
            del seg_probs

    # Step 5: Patch extraction
    patches, patch_info = extract_patches(
        sv_labels, modality_vols, retained_labels, targets,
        n_patch=cfg["n_patch"], patch_neighbors=cfg["patch_neighbors"],
    )

    # Step 6: kNN graph construction
    edge_index, edge_attr, label_to_idx, idx_to_label, graph_info = (
        build_knn_graph(retained_labels, targets, k=cfg["knn_k"])
    )

    # Step 7: Edge-level ground truth
    edge_targets, edge_target_info = compute_edge_targets(
        edge_index, targets, idx_to_label
    )

    return {
        "case_id": case_id,
        "sv_labels": sv_labels,
        "retained_labels": retained_labels,
        "targets": targets,
        "pruning_info": pruning_info,
        "target_summary": target_summary,
        "seg_features": seg_features,
        "seg_summary": seg_summary,
        "patches": patches,
        "patch_info": patch_info,
        "edge_index": edge_index,
        "edge_attr": edge_attr,
        "edge_targets": edge_targets,
        "edge_target_info": edge_target_info,
        "label_to_idx": label_to_idx,
        "idx_to_label": idx_to_label,
        "graph_info": graph_info,
        "modality_vols": modality_vols,
        "gt_seg": gt_seg,
        "shape": shape,
    }

### Test Preprocessing on First Case

In [ ]:
cases = discover_cases(GRAPH["data_root"])
print(f"Discovered {len(cases)} valid BraTS cases")

if cases:
    case = cases[0]
    print(f"Processing: {case['case_id']}")
    t0 = time.time()
    result = preprocess_case(case)
    elapsed = time.time() - t0

    retained = result["retained_labels"]
    print(f"  Time: {elapsed:.1f}s")
    print(f"  Volume shape: {result['shape']}")
    print(f"  Nodes: {len(retained)}, Edges: {result['edge_index'].shape[1]}")
    print(f"  Tumor SVs: {result['target_summary']['tumor_svs']}/{result['target_summary']['total_svs']}")
    print(f"  Patch tensor shape: {result['patch_info']['tensor_shape']}")
    print(f"  Boundary types:")
    for bt, count in sorted(result["edge_target_info"]["boundary_type_distribution"].items()):
        print(f"    {bt}: {count}")
else:
    print("No cases found! Check GRAPH['data_root']")

## Model Architecture### PatchEmbedderConverts per-SV patch tensor (16×19) into 128-dim node embedding via Transformer with modality + position embeddings and [CLS] token pooling.

In [ ]:
# ── Patch-Level Transformer Embedder ─────────────────────────────────


class PatchEmbedder(nn.Module):
    def __init__(self, patch_dim=None, n_modalities=4, n_patch=None,
                 embed_dim=None, n_layers=None, n_heads=None,
                 seg_prior_dim=5, dropout=0.1):
        super().__init__()
        patch_dim = patch_dim or (GRAPH["patch_neighbors"] + 3)
        n_patch = n_patch or GRAPH["n_patch"]
        embed_dim = embed_dim or GRAPH["embed_dim"]
        n_layers = n_layers or GRAPH["transformer_layers"]
        n_heads = n_heads or GRAPH["transformer_heads"]

        self.embed_dim = embed_dim
        self.n_modalities = n_modalities
        self.n_patch = n_patch
        self.n_rows = n_patch * n_modalities  # 16 rows per SV

        # Linear projection: patch row → embed_dim
        self.input_proj = nn.Linear(patch_dim, embed_dim)

        # Learnable modality embeddings (shared across patches of same modality)
        self.modality_embed = nn.Embedding(n_modalities, embed_dim)

        # Learnable patch position embeddings
        self.patch_pos_embed = nn.Embedding(n_patch, embed_dim)

        # [CLS] token
        self.cls_token = nn.Parameter(torch.randn(1, 1, embed_dim) * 0.02)

        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=n_heads,
            dim_feedforward=embed_dim * 4, dropout=dropout,
            batch_first=True, activation="gelu",
        )
        self.transformer = nn.TransformerEncoder(
            encoder_layer, num_layers=n_layers,
        )

        # Seg prior projection (optional, 0-dim if no seg priors)
        self.has_seg_prior = seg_prior_dim > 0
        if self.has_seg_prior:
            self.seg_proj = nn.Sequential(
                nn.Linear(seg_prior_dim, embed_dim),
                nn.GELU(),
            )
            # Final MLP: concat([CLS], seg_embed) → embed_dim
            self.out_proj = nn.Sequential(
                nn.Linear(embed_dim * 2, embed_dim),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(embed_dim, embed_dim),
            )
        else:
            self.out_proj = nn.Sequential(
                nn.Linear(embed_dim, embed_dim),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(embed_dim, embed_dim),
            )

    def forward(self, patch_tensors, seg_priors=None, return_attention=False):
        B = patch_tensors.size(0)
        device = patch_tensors.device

        # Project patch rows
        x = self.input_proj(patch_tensors)  # (B, n_rows, embed_dim)

        # Add modality embeddings: rows are ordered as
        # [patch0_mod0, patch0_mod1, ..., patch0_modM, patch1_mod0, ...]
        mod_ids = torch.arange(self.n_modalities, device=device)
        mod_ids = mod_ids.repeat(self.n_patch)  # (n_rows,)
        x = x + self.modality_embed(mod_ids).unsqueeze(0)  # broadcast over B

        # Add patch position embeddings
        patch_ids = torch.arange(self.n_patch, device=device)
        patch_ids = patch_ids.repeat_interleave(self.n_modalities)  # (n_rows,)
        x = x + self.patch_pos_embed(patch_ids).unsqueeze(0)

        # Prepend [CLS] token
        cls = self.cls_token.expand(B, -1, -1)  # (B, 1, embed_dim)
        x = torch.cat([cls, x], dim=1)  # (B, 1+n_rows, embed_dim)

        # Capture attention weights via manual layer-by-layer forward
        patch_attentions = None
        if return_attention:
            patch_attentions = []
            for layer in self.transformer.layers:
                # Manual forward through TransformerEncoderLayer
                # to capture self-attention weights
                x2, attn_w = layer.self_attn(
                    layer.norm1(x), layer.norm1(x), layer.norm1(x),
                    need_weights=True, average_attn_weights=False,
                )
                patch_attentions.append(attn_w.detach())
                # Complete the layer forward
                x = x + layer.dropout1(x2)
                x = x + layer._ff_block(layer.norm2(x))
        else:
            # Standard forward
            x = self.transformer(x)  # (B, 1+n_rows, embed_dim)

        cls_out = x[:, 0]  # (B, embed_dim)

        # Combine with seg prior if available
        if self.has_seg_prior and seg_priors is not None:
            seg_embed = self.seg_proj(seg_priors)  # (B, embed_dim)
            combined = torch.cat([cls_out, seg_embed], dim=-1)
            node_embeds = self.out_proj(combined)
        else:
            node_embeds = self.out_proj(cls_out)

        return node_embeds, patch_attentions

### GraphEncoder3-layer GATv2 with Laplacian PE, residual connections, and multi-scale fusion.

In [ ]:
# ── Graph-Level GATv2 Encoder ────────────────────────────────────────


class GraphEncoder(nn.Module):
    def __init__(self, embed_dim=None, n_layers=None, n_heads=None,
                 edge_dim=4, pe_dim=None, dropout=0.1):
        super().__init__()
        from torch_geometric.nn import GATv2Conv, LayerNorm

        embed_dim = embed_dim or GRAPH["embed_dim"]
        n_layers = n_layers or GRAPH["gat_layers"]
        n_heads = n_heads or GRAPH["gat_heads"]
        pe_dim = pe_dim or GRAPH["laplacian_pe_dim"]

        self.embed_dim = embed_dim
        self.pe_dim = pe_dim

        # Laplacian PE projection
        self.pe_proj = nn.Linear(pe_dim, embed_dim)

        # GATv2 layers with residual connections
        self.convs = nn.ModuleList()
        self.norms = nn.ModuleList()
        for _ in range(n_layers):
            self.convs.append(GATv2Conv(
                embed_dim, embed_dim // n_heads, heads=n_heads,
                edge_dim=edge_dim, dropout=dropout, concat=True,
            ))
            self.norms.append(LayerNorm(embed_dim))

        # Multiscale fusion: concat all layer outputs → MLP → embed_dim
        self.fusion = nn.Sequential(
            nn.Linear(embed_dim * n_layers, embed_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim, embed_dim),
        )

        self.n_layers = n_layers
        self.dropout = dropout

    def forward(self, x, edge_index, edge_attr=None, lap_pe=None,
                return_attention=False):
        # Add Laplacian PE
        if lap_pe is not None:
            x = x + self.pe_proj(lap_pe)

        layer_outputs = []
        alphas = []

        for i, (conv, norm) in enumerate(zip(self.convs, self.norms)):
            residual = x
            if return_attention:
                x, (_, alpha) = conv(
                    x, edge_index, edge_attr=edge_attr,
                    return_attention_weights=True,
                )
                alphas.append(alpha)
            else:
                x = conv(x, edge_index, edge_attr=edge_attr)
            x = norm(x)
            x = x + residual  # residual connection
            if i < self.n_layers - 1:
                x = F.elu(x)
                x = F.dropout(x, p=self.dropout, training=self.training)
            layer_outputs.append(x)

        # Multiscale fusion
        fused = torch.cat(layer_outputs, dim=-1)  # (N, embed_dim * n_layers)
        out = self.fusion(fused)  # (N, embed_dim)

        return out, alphas

### NodeHead + EdgeHeadBinary tumor classification (Task 1) and 10-class boundary type prediction (Task 2).

In [ ]:
# ── Task 1: Node Classification Head ────────────────────────────────


class NodeHead(nn.Module):
    def __init__(self, embed_dim=None, dropout=0.2):
        super().__init__()
        embed_dim = embed_dim or GRAPH["embed_dim"]
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, 64),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        return self.mlp(x).squeeze(-1)


# ── Task 2: Edge Boundary Type Classification Head ──────────────────


class EdgeHead(nn.Module):
    def __init__(self, embed_dim=None, edge_dim=4, n_boundary_types=10,
                 dropout=0.2):
        super().__init__()
        embed_dim = embed_dim or GRAPH["embed_dim"]
        in_dim = embed_dim * 3 + edge_dim  # h_i + h_j + |h_i-h_j| + e_ij

        self.mlp = nn.Sequential(
            nn.Linear(in_dim, embed_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim, 64),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(64, n_boundary_types),
        )

    def forward(self, node_feats, edge_index, edge_attr):
        h_i = node_feats[edge_index[0]]       # (E, embed_dim)
        h_j = node_feats[edge_index[1]]       # (E, embed_dim)
        h_diff = torch.abs(h_i - h_j)         # (E, embed_dim)

        x = torch.cat([h_i, h_j, h_diff, edge_attr], dim=-1)  # (E, 3*D+4)
        return self.mlp(x)  # (E, 10)

In [ ]:
# ── Laplacian Positional Encoding ────────────────────────────────────


def compute_laplacian_pe(edge_index, num_nodes, k=None):
    from scipy.sparse import coo_matrix
    from scipy.sparse.linalg import eigsh

    k = k or GRAPH["laplacian_pe_dim"]

    if isinstance(edge_index, torch.Tensor):
        edge_index = edge_index.cpu().numpy()

    row = edge_index[0]
    col = edge_index[1]
    data = np.ones(len(row), dtype=np.float64)

    # Build adjacency matrix
    A = coo_matrix((data, (row, col)), shape=(num_nodes, num_nodes)).tocsr()

    # Degree matrix
    deg = np.array(A.sum(axis=1)).flatten()
    deg_inv_sqrt = np.where(deg > 0, 1.0 / np.sqrt(deg), 0.0)

    # Normalized Laplacian: I - D^{-1/2} A D^{-1/2}
    from scipy.sparse import diags
    D_inv_sqrt = diags(deg_inv_sqrt)
    L = diags(np.ones(num_nodes)) - D_inv_sqrt @ A @ D_inv_sqrt

    # Compute k+1 smallest eigenvectors (skip trivial eigenvector 0)
    num_eig = min(k + 1, num_nodes - 1)
    if num_eig < 2:
        return torch.zeros(num_nodes, k, dtype=torch.float32)

    try:
        eigenvalues, eigenvectors = eigsh(L, k=num_eig, which="SM")
        # Skip first eigenvector (constant, eigenvalue ≈ 0)
        pe = eigenvectors[:, 1:k+1]

        # Pad if fewer eigenvectors than k
        if pe.shape[1] < k:
            pad = np.zeros((num_nodes, k - pe.shape[1]))
            pe = np.hstack([pe, pad])

        # Random sign flip for invariance
        signs = np.sign(pe[0])
        signs[signs == 0] = 1
        pe = pe * signs

    except Exception:
        pe = np.zeros((num_nodes, k))

    return torch.from_numpy(pe).float()

### TumorRefinerFull model combining all components.

In [ ]:
# ── Full Model ───────────────────────────────────────────────────────


class TumorRefiner(nn.Module):
    def __init__(self, config=None, use_seg_prior=True):
        super().__init__()
        cfg = config or GRAPH
        seg_dim = 5 if use_seg_prior else 0

        self.embedder = PatchEmbedder(
            patch_dim=cfg["patch_neighbors"] + 3,
            n_modalities=len(cfg["modalities"]),
            n_patch=cfg["n_patch"],
            embed_dim=cfg["embed_dim"],
            n_layers=cfg["transformer_layers"],
            n_heads=cfg["transformer_heads"],
            seg_prior_dim=seg_dim,
        )
        self.encoder = GraphEncoder(
            embed_dim=cfg["embed_dim"],
            n_layers=cfg["gat_layers"],
            n_heads=cfg["gat_heads"],
            edge_dim=4,  # [dx, dy, dz, dist]
            pe_dim=cfg["laplacian_pe_dim"],
        )
        self.node_head = NodeHead(embed_dim=cfg["embed_dim"])
        self.edge_head = EdgeHead(embed_dim=cfg["embed_dim"], edge_dim=4)
        self.use_seg_prior = use_seg_prior

    def forward(self, patch_tensors, seg_priors, edge_index, edge_attr,
                lap_pe=None, return_attention=False):
        # Stage 1: Patch-level embedding
        sp = seg_priors if self.use_seg_prior else None
        node_feats, patch_attns = self.embedder(
            patch_tensors, sp, return_attention=return_attention,
        )

        # Stage 2: Graph-level encoding
        graph_feats, graph_attns = self.encoder(
            node_feats, edge_index, edge_attr=edge_attr,
            lap_pe=lap_pe, return_attention=return_attention,
        )

        # Stage 3: Node classification (Task 1)
        node_logits = self.node_head(graph_feats)  # (N,)

        # Stage 4: Edge boundary classification (Task 2)
        edge_logits = self.edge_head(graph_feats, edge_index, edge_attr)

        attention_dict = {
            "graph": graph_attns,   # list of (E, n_heads) per GATv2 layer
            "patch": patch_attns,   # list of (B, n_heads, seq, seq) per TF layer
        }

        return node_logits, edge_logits, attention_dict

    def count_parameters(self):
        """Count total and per-component trainable parameters."""
        components = {
            "PatchEmbedder": self.embedder,
            "GraphEncoder": self.encoder,
            "NodeHead": self.node_head,
            "EdgeHead": self.edge_head,
        }
        total = 0
        breakdown = {}
        for name, module in components.items():
            n = sum(p.numel() for p in module.parameters() if p.requires_grad)
            breakdown[name] = n
            total += n
        breakdown["Total"] = total
        return breakdown

### Sanity Check — Synthetic Forward Pass

In [ ]:
model = TumorRefiner(use_seg_prior=True).to(DEVICE)
params = model.count_parameters()
print("Parameter count:")
for name, n in params.items():
    print(f"  {name}: {n:,}")

# Synthetic data
N, E = 100, 400
patch_tensors = torch.randn(N, 16, 19, device=DEVICE)
seg_priors = torch.rand(N, 5, device=DEVICE)
edge_index = torch.randint(0, N, (2, E), device=DEVICE)
edge_attr = torch.randn(E, 4, device=DEVICE)
lap_pe = torch.randn(N, 8, device=DEVICE)

model.eval()
with torch.no_grad():
    nl, el, attn = model(patch_tensors, seg_priors, edge_index, edge_attr, lap_pe=lap_pe, return_attention=True)

print(f"\nNode logits: {nl.shape}, prob range: [{torch.sigmoid(nl).min():.4f}, {torch.sigmoid(nl).max():.4f}]")
print(f"Edge logits: {el.shape}")
print(f"GATv2 attention layers: {len(attn['graph'])}, Patch attention layers: {len(attn['patch'])}")
print(f"Model memory: {sum(p.numel() * p.element_size() for p in model.parameters()) / 1e6:.1f} MB")

### Real Data Test

In [ ]:
if cases:
    result = preprocess_case(cases[0])
    retained = result["retained_labels"]
    N_real = len(retained)

    # Assemble tensors
    patch_batch = torch.stack([torch.from_numpy(result["patches"][l]) for l in retained]).to(DEVICE)
    seg_batch = torch.zeros(N_real, 5, device=DEVICE)
    if result["seg_features"] is not None:
        seg_list = []
        for l in retained:
            sf = result["seg_features"][l]
            seg_list.append(torch.from_numpy(np.concatenate([sf["seg_feat"], [sf["seg_entropy"]]])))
        seg_batch = torch.stack(seg_list).float().to(DEVICE)

    ei = torch.from_numpy(result["edge_index"]).to(DEVICE)
    ea = torch.from_numpy(result["edge_attr"]).to(DEVICE)
    lpe = compute_laplacian_pe(result["edge_index"], N_real).to(DEVICE)

    print(f"Case: {result['case_id']}, Nodes: {N_real}, Edges: {ei.shape[1]}")

    model.eval()
    with torch.no_grad():
        nl, el, _ = model(patch_batch, seg_batch, ei, ea, lap_pe=lpe)

    gt_node = torch.tensor([result["targets"][l]["y_cls"] for l in retained], device=DEVICE)
    print(f"Node: pred tumor={(torch.sigmoid(nl)>0.5).sum().item()}, GT tumor={gt_node.sum().item()}")

    gt_edge = torch.from_numpy(result["edge_targets"]["y_edge_type"]).to(DEVICE)
    acc = (el.argmax(-1) == gt_edge).float().mean()
    print(f"Edge: random-init accuracy={acc:.1%}")
    print("End-to-end verified ✓")

## TrainingAdamW optimizer with cosine annealing. Loss = λ_node · BCE(nodes) + λ_edge · CE(edges).

In [ ]:
def assemble_tensors(result, device):
    retained = result["retained_labels"]
    N = len(retained)

    patch_batch = torch.stack([torch.from_numpy(result["patches"][l]) for l in retained]).to(device)

    if result["seg_features"] is not None:
        seg_list = [torch.from_numpy(np.concatenate([result["seg_features"][l]["seg_feat"],
                    [result["seg_features"][l]["seg_entropy"]]]))for l in retained]
        seg_batch = torch.stack(seg_list).float().to(device)
    else:
        seg_batch = torch.zeros(N, 5, device=device)

    ei = torch.from_numpy(result["edge_index"]).to(device)
    ea = torch.from_numpy(result["edge_attr"]).to(device)
    lpe = compute_laplacian_pe(result["edge_index"], N).to(device)

    gt_node = torch.tensor([result["targets"][l]["y_cls"] for l in retained],
                           dtype=torch.float32, device=device)
    gt_edge = torch.from_numpy(result["edge_targets"]["y_edge_type"]).to(device)

    return patch_batch, seg_batch, ei, ea, lpe, gt_node, gt_edge


def train_one_epoch(model, graphs, optimizer, scheduler, lambda_node=1.0, lambda_edge=0.5, accum_steps=4):
    model.train()
    total_loss, n_node_correct, n_node_total = 0.0, 0, 0
    n_edge_correct, n_edge_total = 0, 0
    optimizer.zero_grad()

    for i, result in enumerate(graphs):
        patches, seg, ei, ea, lpe, gt_node, gt_edge = assemble_tensors(result, DEVICE)
        nl, el, _ = model(patches, seg, ei, ea, lap_pe=lpe)

        loss_node = F.binary_cross_entropy_with_logits(nl, gt_node)
        loss_edge = F.cross_entropy(el, gt_edge)
        loss = lambda_node * loss_node + lambda_edge * loss_edge
        loss = loss / accum_steps
        loss.backward()

        if (i + 1) % accum_steps == 0 or (i + 1) == len(graphs):
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

        total_loss += loss.item() * accum_steps
        preds_node = (torch.sigmoid(nl) > 0.5).long()
        n_node_correct += (preds_node == gt_node.long()).sum().item()
        n_node_total += len(gt_node)
        n_edge_correct += (el.argmax(-1) == gt_edge).sum().item()
        n_edge_total += len(gt_edge)

    return {
        "loss": total_loss / len(graphs),
        "node_acc": n_node_correct / max(n_node_total, 1),
        "edge_acc": n_edge_correct / max(n_edge_total, 1),
    }


@torch.no_grad()
def evaluate(model, graphs):
    model.eval()
    total_loss, n_node_correct, n_node_total = 0.0, 0, 0
    n_edge_correct, n_edge_total = 0, 0

    for result in graphs:
        patches, seg, ei, ea, lpe, gt_node, gt_edge = assemble_tensors(result, DEVICE)
        nl, el, _ = model(patches, seg, ei, ea, lap_pe=lpe)

        loss_node = F.binary_cross_entropy_with_logits(nl, gt_node)
        loss_edge = F.cross_entropy(el, gt_edge)
        loss = GRAPH["lambda_node"] * loss_node + GRAPH["lambda_edge"] * loss_edge

        total_loss += loss.item()
        preds_node = (torch.sigmoid(nl) > 0.5).long()
        n_node_correct += (preds_node == gt_node.long()).sum().item()
        n_node_total += len(gt_node)
        n_edge_correct += (el.argmax(-1) == gt_edge).sum().item()
        n_edge_total += len(gt_edge)

    return {
        "loss": total_loss / max(len(graphs), 1),
        "node_acc": n_node_correct / max(n_node_total, 1),
        "edge_acc": n_edge_correct / max(n_edge_total, 1),
    }

print("Training functions defined ✓")

In [ ]:
# Preprocess all cases (or subset)
MAX_CASES = 10  # adjust based on available time/memory
cases_to_use = cases[:MAX_CASES]

print(f"Preprocessing {len(cases_to_use)} cases...")
all_graphs = []
for i, case in enumerate(cases_to_use):
    t0 = time.time()
    result = preprocess_case(case)
    all_graphs.append(result)
    print(f"  [{i+1}/{len(cases_to_use)}] {case['case_id']}: "
          f"{len(result['retained_labels'])} nodes, {result['edge_index'].shape[1]} edges "
          f"({time.time()-t0:.1f}s)")

# Split: 80% train, 20% val
split = int(0.8 * len(all_graphs))
train_graphs = all_graphs[:split]
val_graphs = all_graphs[split:]
print(f"\nTrain: {len(train_graphs)} graphs, Val: {len(val_graphs)} graphs")

# Train
model = TumorRefiner(use_seg_prior=True).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=GRAPH["lr"], weight_decay=GRAPH["weight_decay"])
total_steps = GRAPH["epochs"] * len(train_graphs) // GRAPH["accum_steps"]
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps)

print(f"\nTraining for {GRAPH['epochs']} epochs...")
for epoch in range(1, GRAPH["epochs"] + 1):
    random.shuffle(train_graphs)
    train_metrics = train_one_epoch(model, train_graphs, optimizer, scheduler,
                                    GRAPH["lambda_node"], GRAPH["lambda_edge"], GRAPH["accum_steps"])

    if epoch % GRAPH["eval_every"] == 0 or epoch == 1:
        val_metrics = evaluate(model, val_graphs)
        print(f"Epoch {epoch:3d} | "
              f"Train loss={train_metrics['loss']:.4f} node={train_metrics['node_acc']:.1%} edge={train_metrics['edge_acc']:.1%} | "
              f"Val loss={val_metrics['loss']:.4f} node={val_metrics['node_acc']:.1%} edge={val_metrics['edge_acc']:.1%}")

# Save checkpoint
Path("checkpoints").mkdir(exist_ok=True)
torch.save(model.state_dict(), str(GRAPH["checkpoint"]))
print(f"\nModel saved to {GRAPH['checkpoint']}")

## Explainability (3 Levels)1. **Graph Attention:** Which neighbors influenced this node's prediction?2. **Patch Attention:** Which MRI patches/modalities did the Transformer focus on?3. **Refinement:** Where did the GNN correct/degrade the seg model?

In [ ]:
# ── Graph-Level Attention Traces ─────────────────────────────────────


def graph_attention_trace(node_idx, edge_index, graph_attentions,
                          targets=None, idx_to_label=None, top_k=5):
    if isinstance(edge_index, np.ndarray):
        edge_index = torch.from_numpy(edge_index)

    # Find all edges where node_idx is the destination
    dst_mask = (edge_index[1] == node_idx)
    incoming_indices = torch.where(dst_mask)[0]

    if len(incoming_indices) == 0:
        return {
            "node_idx": node_idx,
            "sv_label": idx_to_label.get(node_idx) if idx_to_label else None,
            "neighbors": [],
            "attention_entropy": 0.0,
            "total_incoming_edges": 0,
        }

    # Source nodes for incoming edges
    src_nodes = edge_index[0, incoming_indices].cpu().numpy()
    unique_src = np.unique(src_nodes)

    # Aggregate attention per source node across layers and heads
    neighbor_attn = {}
    for src in unique_src:
        layer_attns = []
        for layer_idx, attn in enumerate(graph_attentions):
            # attn shape: (E, n_heads) — attention for each edge per head
            # Find edges from src → node_idx
            src_mask = (edge_index[0] == src) & (edge_index[1] == node_idx)
            edge_positions = torch.where(src_mask)[0]
            if len(edge_positions) > 0:
                # Average attention across heads for this edge
                edge_attn = attn[edge_positions].mean().item()
                layer_attns.append(edge_attn)
            else:
                layer_attns.append(0.0)

        neighbor_attn[int(src)] = {
            "per_layer": layer_attns,
            "mean": float(np.mean(layer_attns)),
        }

    # Sort by mean attention (descending)
    ranked = sorted(neighbor_attn.items(), key=lambda x: x[1]["mean"],
                    reverse=True)

    # Compute attention entropy (how focused vs. diffuse)
    attn_values = np.array([v["mean"] for _, v in ranked])
    attn_sum = attn_values.sum()
    if attn_sum > 0:
        attn_probs = attn_values / attn_sum
        attn_probs = attn_probs[attn_probs > 0]
        entropy = float(-np.sum(attn_probs * np.log(attn_probs)))
    else:
        entropy = 0.0

    # Build neighbor list with optional GT context
    neighbors = []
    for src, attn_info in ranked[:top_k]:
        entry = {
            "node_idx": src,
            "sv_label": idx_to_label.get(src) if idx_to_label else None,
            "mean_attention": attn_info["mean"],
            "per_layer_attention": attn_info["per_layer"],
        }
        # Add GT context if available
        if targets and idx_to_label:
            label = idx_to_label.get(src)
            if label and label in targets:
                t = targets[label]
                entry["y_cls"] = t["y_cls"]
                entry["y_dominant"] = t["y_dominant"]
                entry["centroid"] = t["centroid"].tolist()
        neighbors.append(entry)

    return {
        "node_idx": node_idx,
        "sv_label": idx_to_label.get(node_idx) if idx_to_label else None,
        "neighbors": neighbors,
        "attention_entropy": entropy,
        "total_incoming_edges": len(incoming_indices),
    }


# ── Patch-Level Attention Traces ─────────────────────────────────────


def patch_attention_trace(node_idx, patch_attentions, n_modalities=4,
                          n_patch=None, modality_names=None):
    n_patch = n_patch or GRAPH["n_patch"]
    modality_names = modality_names or GRAPH["modalities"]
    n_rows = n_patch * n_modalities

    if not patch_attentions:
        return {
            "node_idx": node_idx,
            "cls_to_patch_attention": np.zeros(n_rows),
            "per_patch_importance": np.zeros(n_patch),
            "per_modality_importance": np.zeros(n_modalities),
            "top_patches": [],
            "attention_layers": 0,
        }

    # Average [CLS]→patch attention across layers and heads
    # Attention shape: (B, n_heads, seq_len, seq_len)
    # seq_len = 1 (CLS) + n_rows (patches)
    # CLS is at position 0
    cls_attn_layers = []
    for attn in patch_attentions:
        # attn[node_idx]: (n_heads, seq_len, seq_len)
        # CLS → patches: row 0, columns 1:n_rows+1
        node_attn = attn[node_idx]  # (n_heads, seq_len, seq_len)
        cls_to_all = node_attn[:, 0, 1:]  # (n_heads, n_rows) — CLS to patches
        cls_attn_layers.append(cls_to_all.cpu().numpy())

    # Average across layers and heads
    avg_attn = np.mean(cls_attn_layers, axis=(0, 1))  # (n_rows,)

    # Aggregate per patch centroid (sum over modalities within each patch)
    per_patch = np.zeros(n_patch)
    for p in range(n_patch):
        start = p * n_modalities
        end = start + n_modalities
        per_patch[p] = avg_attn[start:end].sum()

    # Aggregate per modality (sum over patches within each modality)
    per_mod = np.zeros(n_modalities)
    for m in range(n_modalities):
        indices = [p * n_modalities + m for p in range(n_patch)]
        per_mod[m] = avg_attn[indices].sum()

    # Rank individual patch rows
    ranked_indices = np.argsort(avg_attn)[::-1]
    top_patches = []
    for idx in ranked_indices[:8]:  # top 8 rows
        patch_id = idx // n_modalities
        mod_id = idx % n_modalities
        top_patches.append({
            "row_idx": int(idx),
            "patch_id": int(patch_id),
            "modality_id": int(mod_id),
            "modality_name": modality_names[mod_id] if mod_id < len(modality_names) else f"mod_{mod_id}",
            "attention": float(avg_attn[idx]),
        })

    return {
        "node_idx": node_idx,
        "cls_to_patch_attention": avg_attn,
        "per_patch_importance": per_patch,
        "per_modality_importance": per_mod,
        "modality_names": modality_names,
        "top_patches": top_patches,
        "attention_layers": len(patch_attentions),
    }


# ── Refinement-Level Traces ──────────────────────────────────────────


def refinement_trace(node_logits, targets, retained_labels,
                     seg_features=None, idx_to_label=None,
                     edge_index=None, graph_attentions=None,
                     threshold=0.5):

    N = len(retained_labels)
    gnn_preds = (torch.sigmoid(node_logits) > threshold).cpu().numpy()

    gt_labels = np.array([targets[retained_labels[i]]["y_cls"]
                          for i in range(N)])

    # Seg model predictions (binary: is dominant class != BG?)
    if seg_features is not None:
        seg_preds = np.array([
            1 if seg_features[retained_labels[i]]["seg_pred"] != 0 else 0
            for i in range(N)
        ])
    else:
        seg_preds = np.zeros(N, dtype=int)

    # Accuracy
    gnn_correct = (gnn_preds == gt_labels).sum()
    seg_correct = (seg_preds == gt_labels).sum()
    accuracy_gnn = float(gnn_correct / max(N, 1))
    accuracy_seg = float(seg_correct / max(N, 1))

    # Find corrections and degradations
    seg_wrong = (seg_preds != gt_labels)
    seg_right = (seg_preds == gt_labels)
    gnn_right = (gnn_preds == gt_labels)
    gnn_wrong = (gnn_preds != gt_labels)

    corrections_mask = seg_wrong & gnn_right  # seg wrong, GNN fixed it
    degradations_mask = seg_right & gnn_wrong  # seg right, GNN broke it

    n_seg_errors = seg_wrong.sum()
    correction_rate = float(corrections_mask.sum() / max(n_seg_errors, 1))
    n_seg_correct = seg_right.sum()
    degradation_rate = float(degradations_mask.sum() / max(n_seg_correct, 1))

    # Build correction details
    corrections = []
    for i in range(N):
        if not corrections_mask[i] and not degradations_mask[i]:
            continue

        label = retained_labels[i]
        entry = {
            "node_idx": i,
            "sv_label": label,
            "gt": int(gt_labels[i]),
            "gnn_pred": int(gnn_preds[i]),
            "gnn_prob": float(torch.sigmoid(node_logits[i]).item()),
            "seg_pred": int(seg_preds[i]),
            "type": "correction" if corrections_mask[i] else "degradation",
            "gt_dominant": targets[label]["y_dominant"],
            "centroid": targets[label]["centroid"].tolist(),
        }

        # Add seg prior context
        if seg_features and label in seg_features:
            sf = seg_features[label]
            entry["seg_entropy"] = float(sf["seg_entropy"])
            entry["seg_feat"] = sf["seg_feat"].tolist()

        # Trace back through graph attention for corrections
        if (corrections_mask[i] and edge_index is not None
                and graph_attentions is not None):
            trace = graph_attention_trace(
                i, edge_index, graph_attentions,
                targets=targets, idx_to_label=idx_to_label, top_k=3,
            )
            entry["attention_trace"] = trace["neighbors"]
            entry["attention_entropy"] = trace["attention_entropy"]

        corrections.append(entry)

    return {
        "total_nodes": N,
        "accuracy_gnn": accuracy_gnn,
        "accuracy_seg": accuracy_seg,
        "correction_rate": correction_rate,
        "degradation_rate": degradation_rate,
        "n_corrections": int(corrections_mask.sum()),
        "n_degradations": int(degradations_mask.sum()),
        "corrections": corrections,
    }


# ── Full Explanation Report ──────────────────────────────────────────


def explain_case(model, result, device=None, top_k_nodes=10):
    # compute_laplacian_pe already defined above

    device = device or GRAPH.get("device", "cpu")
    retained = result["retained_labels"]
    N = len(retained)

    # Assemble tensors
    patch_list = [torch.from_numpy(result["patches"][l]) for l in retained]
    patch_batch = torch.stack(patch_list, dim=0).to(device)

    if result["seg_features"] is not None:
        seg_list = []
        for l in retained:
            sf = result["seg_features"][l]
            feat = np.concatenate([sf["seg_feat"], [sf["seg_entropy"]]])
            seg_list.append(torch.from_numpy(feat))
        seg_batch = torch.stack(seg_list, dim=0).float().to(device)
    else:
        seg_batch = torch.zeros(N, 5, device=device)

    ei = torch.from_numpy(result["edge_index"]).to(device)
    ea = torch.from_numpy(result["edge_attr"]).to(device)
    lpe = compute_laplacian_pe(result["edge_index"], N).to(device)

    # Forward with attention
    model.eval()
    with torch.no_grad():
        node_logits, edge_logits, attn_dict = model(
            patch_batch, seg_batch, ei, ea,
            lap_pe=lpe, return_attention=True,
        )

    # ── Level 1: Graph attention traces ──
    # Focus on tumor SVs and high-probability predictions
    probs = torch.sigmoid(node_logits).cpu()
    tumor_indices = [i for i in range(N)
                     if result["targets"][retained[i]]["y_cls"] == 1]
    high_prob_indices = torch.argsort(probs, descending=True)[:top_k_nodes].tolist()
    explain_indices = list(set(tumor_indices + high_prob_indices))[:top_k_nodes]

    graph_traces = {}
    for idx in explain_indices:
        graph_traces[idx] = graph_attention_trace(
            idx, ei, attn_dict["graph"],
            targets=result["targets"],
            idx_to_label=result["idx_to_label"],
        )

    # ── Level 2: Patch attention traces ──
    patch_traces = {}
    if attn_dict["patch"]:
        for idx in explain_indices:
            patch_traces[idx] = patch_attention_trace(
                idx, attn_dict["patch"],
            )

    # ── Level 3: Refinement trace ──
    ref_trace = refinement_trace(
        node_logits, result["targets"], retained,
        seg_features=result["seg_features"],
        idx_to_label=result["idx_to_label"],
        edge_index=ei, graph_attentions=attn_dict["graph"],
    )

    return {
        "case_id": result["case_id"],
        "graph_traces": graph_traces,
        "patch_traces": patch_traces,
        "refinement": ref_trace,
        "node_logits": node_logits.cpu(),
        "edge_logits": edge_logits.cpu(),
    }

### Generate Explainability Report

In [ ]:
if cases:
    # Use first case for explainability demo
    test_result = all_graphs[0] if all_graphs else preprocess_case(cases[0])

    t0 = time.time()
    report = explain_case(model, test_result, device=DEVICE)
    print(f"Explanation generated in {time.time()-t0:.1f}s\n")

    # Level 1: Graph traces
    print("=" * 60)
    print("Level 1: Graph Attention Traces")
    print("=" * 60)
    for idx, trace in list(report["graph_traces"].items())[:3]:
        label = trace["sv_label"]
        gt = test_result["targets"].get(label, {})
        prob = float(torch.sigmoid(report["node_logits"][idx]))
        print(f"\n  Node {idx} (SV {label}): p={prob:.3f}, GT={gt.get('y_cls', '?')}, "
              f"entropy={trace['attention_entropy']:.3f}")
        for nb in trace["neighbors"][:3]:
            nb_gt = f"y_cls={nb.get('y_cls', '?')}" if 'y_cls' in nb else ""
            print(f"    → Neighbor {nb['node_idx']} (SV {nb['sv_label']}): "
                  f"attn={nb['mean_attention']:.4f}, {nb_gt}")

    # Level 2: Patch traces
    print(f"\n{'='*60}")
    print("Level 2: Patch Attention Traces")
    print("=" * 60)
    for idx, trace in list(report["patch_traces"].items())[:2]:
        print(f"\n  Node {idx}: {trace['attention_layers']} layers")
        print(f"    Modality importance: ", end="")
        for m, name in enumerate(trace["modality_names"]):
            print(f"{name}={trace['per_modality_importance'][m]:.3f}", end="  ")
        print()

    # Level 3: Refinement
    ref = report["refinement"]
    print(f"\n{'='*60}")
    print("Level 3: Refinement Trace")
    print("=" * 60)
    print(f"  GNN accuracy:  {ref['accuracy_gnn']:.1%}")
    print(f"  Seg accuracy:  {ref['accuracy_seg']:.1%}")
    print(f"  Corrections:   {ref['n_corrections']} (rate: {ref['correction_rate']:.1%})")
    print(f"  Degradations:  {ref['n_degradations']} (rate: {ref['degradation_rate']:.1%})")

    print(f"\nExplainability pipeline verified ✓")